# 02 — Подготовка датасета для обучения YOLO

Концертация разметки из `Khasan_Dataset/` в формат YOLO bbox и собрка `data/yolo/` для обучения на Kaggle

1. Конвертация смешанного формата разметки в YOLO bbox
2. Деление на train/val (видео `43_15` val, остальные train)
3. `dataset.yaml`


**Input:** `Khasan_Dataset/train/{images,labels}/`  
**Output:** `data/yolo/{images,labels}/{train,val}/`, `data/yolo/dataset.yaml`

In [ ]:
from pathlib import Path

ROOT       = Path.cwd().parent if Path.cwd().name == 'eda' else Path.cwd()
SRC_IMAGES = ROOT / 'Khasan_Dataset/train/images'
SRC_LABELS = ROOT / 'Khasan_Dataset/train/labels'
DST_IMAGES = ROOT / 'data/yolo/images'
DST_LABELS = ROOT / 'data/yolo/labels'
YAML_PATH  = ROOT / 'data/yolo/dataset.yaml'
BEST_PT    = ROOT / 'runs/tags_v1/weights/best.pt'
VAL_PREFIX = '43_15'

print(f'ROOT:           {ROOT}')
print(f'Khasan_Dataset: {SRC_IMAGES.exists()} / {SRC_LABELS.exists()}')
print(f'best.pt:        {BEST_PT}  (exists={BEST_PT.exists()})')

## Конвертация разметки

In [6]:
import os
import shutil

# Чистая пересборка
if DST_LABELS.exists():
    shutil.rmtree(DST_LABELS)

if DST_IMAGES.exists():
    shutil.rmtree(DST_IMAGES)

for split in ['train', 'val']:
    (DST_IMAGES / split).mkdir(parents=True, exist_ok=True)
    (DST_LABELS / split).mkdir(parents=True, exist_ok=True)

n_train = 0
n_val = 0
n_bbox = 0
n_poly = 0

for lbl in sorted(SRC_LABELS.glob('*.txt')):
    if lbl.stem.startswith(VAL_PREFIX):
        split = 'val'
    else:
        split = 'train'

    out_lines = []

    text = lbl.read_text(encoding='utf-8').strip()

    for line in text.splitlines():
        line = line.strip()

        if line == '':
            continue

        parts = line.split()
        class_id = parts[0]

        # Уже готовый YOLO bbox: class cx cy w h
        if len(parts) == 5:
            out_line = ' '.join(parts)
            n_bbox += 1

        # Полигон: class x1 y1 x2 y2 ...
        else:
            coords = list(map(float, parts[1:]))

            xs = coords[0::2]
            ys = coords[1::2]

            x1 = min(xs)
            x2 = max(xs)
            y1 = min(ys)
            y2 = max(ys)

            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            w = x2 - x1
            h = y2 - y1

            out_line = f'{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}'
            n_poly += 1

        out_lines.append(out_line)

    # Сохраняем label
    dst_label = DST_LABELS / split / lbl.name
    dst_label.write_text('\n'.join(out_lines), encoding='utf-8')

    # Ищем картинку
    img = None

    for ext in ['.jpg', '.jpeg', '.png']:
        candidate = SRC_IMAGES / f'{lbl.stem}{ext}'

        if candidate.exists():
            img = candidate
            break

    if img is None:
        continue

    # Создаём ссылку на картинку
    dst_img = DST_IMAGES / split / img.name

    if not dst_img.exists():
        os.symlink(img.resolve(), dst_img)

    if split == 'val':
        n_val += 1
    else:
        n_train += 1

print(f'train: {n_train} изображений   val: {n_val}')
print(f'Разметок: {n_bbox} bbox (5-token) + {n_poly} полигонов → {n_bbox + n_poly} итого')

train: 256 изображений   val: 45
Разметок: 2659 bbox (5-token) + 49 полигонов → 2708 итого


## Запись `dataset.yaml`

In [7]:
YAML_PATH.write_text(
    f"path: {ROOT / 'data/yolo'}\n"
    "train: images/train\n"
    "val: images/val\n\n"
    "nc: 1\n"
    "names: ['price_tag']\n"
)
print(YAML_PATH.read_text())

path: /Users/alex/ML Projects/Lenta Tech Life Hack/data/yolo
train: images/train
val: images/val

nc: 1
names: ['price_tag']



In [ ]:
import cv2
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

SMOKE_VIDEO = ROOT / 'Данные/Unlabeled/26_2-10.mp4'
SMOKE_TS_MS = 1000
SMOKE_CONF  = 0.25

# Авто-выбор устройства: cuda → mps → cpu
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'DEVICE: {DEVICE}')

assert BEST_PT.exists(), f'best.pt не найден в {BEST_PT} — сначала скачай с Kaggle'

# Захватываем кадр
cap = cv2.VideoCapture(str(SMOKE_VIDEO))
cap.set(cv2.CAP_PROP_POS_MSEC, float(SMOKE_TS_MS))
ok, frame = cap.read()
cap.release()
assert ok, 'Не удалось захватить кадр'
H, W = frame.shape[:2]

# Инференс
model = YOLO(str(BEST_PT))
result = model(frame, imgsz=1280, conf=SMOKE_CONF, device=DEVICE, verbose=False)[0]
n_det = len(result.boxes) if result.boxes is not None else 0
print(f'Детекций (conf≥{SMOKE_CONF}): {n_det}')

# Рисуем
annotated = cv2.cvtColor(frame.copy(), cv2.COLOR_BGR2RGB)
for box in (result.boxes or []):
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    conf = float(box.conf[0])
    cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 220, 0), 6)
    cv2.putText(annotated, f'price_tag {conf:.2f}', (x1, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX, max(1.5, W / 1280),
                (0, 220, 0), max(3, int(W / 640)))

plt.figure(figsize=(14, 8))
plt.imshow(cv2.resize(annotated, (W // 3, H // 3), interpolation=cv2.INTER_AREA))
plt.title(f'Smoke-тест: {n_det} детекций (conf≥{SMOKE_CONF})')
plt.axis('off')
plt.tight_layout()
plt.show()